# Notebook 03: Data Preprocessing
##### Football Match Prediction 
##### Student Name: Vishal Chaudhary
##### Student Number: X23332794

### IMPORT REQUIRED LIBRARIES

In [43]:
# IMPORT REQUIRED LIBRARIES
import pandas as pd
import numpy as np
import gc
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Libraries imported successfully!")
print(f"Processing Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Libraries imported successfully!
Processing Date: 2025-11-14 00:24:55



### LOAD DATASETS

In [44]:

# Define paths
DATA_RAW_PATH = Path('/Users/vishalchaudhary/Desktop/Football_Project_Final/DOMAIN_APPLICATIONS_PROJECT/data/raw')
PROCESSED_PATH = Path('/Users/vishalchaudhary/Desktop/Football_Project_Final/DOMAIN_APPLICATIONS_PROJECT/data/processed')
FEATURES_PATH = Path('/Users/vishalchaudhary/Desktop/Football_Project_Final/DOMAIN_APPLICATIONS_PROJECT/data/features')
RESULTS_PATH = Path('/Users/vishalchaudhary/Desktop/Football_Project_Final/DOMAIN_APPLICATIONS_PROJECT/results')
FIGURES_PATH = RESULTS_PATH / 'figures'

FEATURES_PATH.mkdir(parents=True, exist_ok=True)

print("LOADING PROCESSED DATASETS\n")

# Load processed data from Notebook 02
df_fixtures = pd.read_csv(PROCESSED_PATH / 'fixtures_with_target.csv')

# Load Raw datasets
df_historical_fixtures_raw = pd.read_csv(DATA_RAW_PATH / 'historical_fixtures_2018_2023_20251101_203250.csv')
df_top_players_raw = pd.read_csv(DATA_RAW_PATH / 'top_500_player_stats_2018_2023.csv')
df_injuries_raw = pd.read_csv(DATA_RAW_PATH / 'injuries_data_2018_2023_20251105_003341.csv')
df_squad_basic_raw = pd.read_csv(DATA_RAW_PATH / 'squad_basic_20251103_123202.csv')
df_standings_raw = pd.read_csv(DATA_RAW_PATH / 'league_standings_20251101_215708.csv')
df_league_teams_raw = pd.read_csv(DATA_RAW_PATH / 'league_teams_20251101_215613.csv')
df_h2h_raw = pd.read_csv(DATA_RAW_PATH / 'h2h_matches_20251101_224848.csv')
df_fixture_stats_raw = pd.read_csv(DATA_RAW_PATH / 'fixture_statistics_complete.csv')
df_lineups_raw = pd.read_csv(DATA_RAW_PATH/'lineups_complete.csv')


LOADING PROCESSED DATASETS



In [45]:
print(f"Fixtures columns: {df_fixtures.columns.tolist()[:]}\n")
print(f"Fixtures columns: {df_historical_fixtures_raw.columns.tolist()[:]}\n")

Fixtures columns: ['fixture_id', 'date', 'timestamp', 'league_id', 'league_name', 'season', 'round', 'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name', 'home_goals', 'away_goals', 'home_score_halftime', 'away_score_halftime', 'home_score_fulltime', 'away_score_fulltime', 'venue_name', 'venue_city', 'referee', 'status', 'match_outcome', 'target', 'total_goals', 'goal_diff', 'home_clean_sheet', 'away_clean_sheet', 'both_scored', 'year', 'month', 'month_name', 'day_of_week']

Fixtures columns: ['fixture_id', 'date', 'timestamp', 'league_id', 'league_name', 'season', 'round', 'home_team_id', 'home_team_name', 'away_team_id', 'away_team_name', 'home_goals', 'away_goals', 'home_score_halftime', 'away_score_halftime', 'home_score_fulltime', 'away_score_fulltime', 'venue_name', 'venue_city', 'referee', 'status']



### Perform quality check on a dataframe

In [46]:
# Function to perform quality check on a dataframe
def data_quality_check(df, name):
    print(f"\\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\\nData Types:\n{df.dtypes}\n")
    print(f"\\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\\nSample Data (first 5 rows):\n{df.head(5)}\n")
    print(f"\\nUnique Values in Key Columns:")
    key_cols = df.columns[:5]  # First 5 columns as example; adjust if needed
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"f\\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\\n")

# Run quality checks for each dataframe
data_quality_check(df_fixtures, "df_fixtures")
data_quality_check(df_historical_fixtures_raw, "df_historical_fixtures_raw")
data_quality_check(df_top_players_raw, "df_top_players_raw")
data_quality_check(df_injuries_raw, "df_injuries_raw")
data_quality_check(df_squad_basic_raw, "df_squad_basic_raw")
data_quality_check(df_standings_raw, "df_standings_raw")
data_quality_check(df_league_teams_raw, "df_league_teams_raw")
data_quality_check(df_h2h_raw, "df_h2h_raw")
data_quality_check(df_fixture_stats_raw, "df_fixture_stats_raw")
data_quality_check(df_lineups_raw, "df_lineups_raw")

\n Quality Check for df_fixtures 

Shape: (18894, 32)
\nData Types:
fixture_id               int64
date                    object
timestamp                int64
league_id                int64
league_name             object
season                   int64
round                   object
home_team_id             int64
home_team_name          object
away_team_id             int64
away_team_name          object
home_goals               int64
away_goals               int64
home_score_halftime    float64
away_score_halftime    float64
home_score_fulltime      int64
away_score_fulltime      int64
venue_name              object
venue_city              object
referee                 object
status                  object
match_outcome           object
target                   int64
total_goals              int64
goal_diff                int64
home_clean_sheet         int64
away_clean_sheet         int64
both_scored              int64
year                     int64
month                    int64
mo

### Data Quality Improvement for df_fixtures 

In [47]:
# Data Quality Improvement for df_fixtures 

# 1. Convert date to datetime
df_fixtures['date'] = pd.to_datetime(df_fixtures['date'], errors='coerce')

# 2. Check and handle missing values
print("\nMissing Values Before Handling:\n", df_fixtures.isna().sum())

# Define columns by type for imputation
numeric_cols = ['home_goals', 'away_goals', 'home_score_halftime', 'away_score_halftime', 
                'home_score_fulltime', 'away_score_fulltime', 'total_goals', 'goal_diff', 
                'home_clean_sheet', 'away_clean_sheet', 'both_scored', 'year', 'month']
categorical_cols = ['venue_name', 'venue_city', 'referee', 'status', 'match_outcome', 'month_name', 'day_of_week']

# Impute numeric columns with median
for col in numeric_cols:
    if col in df_fixtures.columns:
        df_fixtures[col].fillna(df_fixtures[col].median(), inplace=True)

# Impute categorical columns with mode or 'Unknown'
for col in categorical_cols:
    if col in df_fixtures.columns:
        df_fixtures[col].fillna(df_fixtures[col].mode()[0] if not df_fixtures[col].mode().empty else 'Unknown', inplace=True)

# Handle any remaining missing dates (unlikely, but for robustness)
df_fixtures['date'].fillna(df_fixtures['date'].mode()[0], inplace=True)

# 3. Drop duplicates (if any)
df_fixtures.drop_duplicates(inplace=True)

# 4. Ensure correct data types
type_corrections = {
    'home_clean_sheet': 'int32',
    'away_clean_sheet': 'int32',
    'both_scored': 'int32',
    'year': 'int32',
    'month': 'int32',
    'fixture_id': 'int64',
    'league_id': 'int64',
    'home_team_id': 'int64',
    'away_team_id': 'int64'
}
for col, dtype in type_corrections.items():
    if col in df_fixtures.columns:
        df_fixtures[col] = df_fixtures[col].astype(dtype)

# 5. Re-run quality check
print("\n Quality Check for df_fixtures (After Cleaning) \n")
print(f"Shape: {df_fixtures.shape}\n")
print(f"Data Types:\n{df_fixtures.dtypes}\n")
print(f"Missing Values per Column:\n{df_fixtures.isna().sum()}\n")
total_missing = df_fixtures.isna().sum().sum()
print(f"Total Missing Values: {total_missing} ({total_missing / (df_fixtures.shape[0] * df_fixtures.shape[1]) * 100:.2f}% of data)\n")
print(f"Number of Duplicates: {df_fixtures.duplicated().sum()}\n")
print(f"Unique Values in Key Columns:")
key_cols = ['fixture_id', 'league_id', 'home_team_id', 'away_team_id', 'season']
for col in key_cols:
    print(f"{col}: {df_fixtures[col].nunique()} unique values")
print("\nDescriptive Statistics for Numeric Columns:\n", df_fixtures[numeric_cols].describe(), "\n")



Missing Values Before Handling:
 fixture_id               0
date                     0
timestamp                0
league_id                0
league_name              0
season                   0
round                    0
home_team_id             0
home_team_name           0
away_team_id             0
away_team_name           0
home_goals               0
away_goals               0
home_score_halftime      7
away_score_halftime      7
home_score_fulltime      0
away_score_fulltime      0
venue_name               2
venue_city              80
referee                206
status                   0
match_outcome            0
target                   0
total_goals              0
goal_diff                0
home_clean_sheet         0
away_clean_sheet         0
both_scored              0
year                     0
month                    0
month_name               0
day_of_week              0
dtype: int64

 Quality Check for df_fixtures (After Cleaning) 

Shape: (18894, 32)

Data Types:
fixtur

### Data Cleaning for All Datasets

In [48]:
#  Data Cleaning for All Datasets

def clean_dataframe(df, name, numeric_cols=None, categorical_cols=None, date_cols=None, drop_cols=None, drop_duplicates=False):
    print(f"\nCleaning {name}...")
    # Make a copy to avoid modifying original
    df = df.copy()
    
    # Drop specified columns
    if drop_cols:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
    
    # Convert dates
    if date_cols:
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Impute numerics with median
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
    
    # Impute categoricals with mode or 'Unknown'
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    # Drop duplicates if specified
    if drop_duplicates:
        df.drop_duplicates(inplace=True)
    
    # Re-run quality check
    data_quality_check(df, name)
    return df

# Cleaning each dataset
# 1. df_historical_fixtures_raw 

# 2. df_top_players_raw
df_top_players_raw = clean_dataframe(
    df_top_players_raw, "df_top_players_raw",
    numeric_cols=['age', 'minutes', 'rating', 'goals_total', 'assists', 'shots_total', 'shots_on', 
                  'passes_total', 'key_passes', 'tackles_total', 'interceptions', 'duels_total', 
                  'duels_won', 'dribbles_success', 'fouls_committed'],
    categorical_cols=['nationality', 'team', 'position', 'player_name'],
    date_cols=None,
    drop_cols=['appearances'],  # 100% missing
    drop_duplicates=False
)

# 3. df_injuries_raw
df_injuries_raw = clean_dataframe(
    df_injuries_raw, "df_injuries_raw",
    numeric_cols=None,
    categorical_cols=['player_name', 'injury_reason', 'team'],
    date_cols=['fixture_date'],
    drop_cols=['age'],  # 100% missing
    drop_duplicates=True
)

# 4. df_squad_basic_raw
df_squad_basic_raw = clean_dataframe(
    df_squad_basic_raw, "df_squad_basic_raw",
    numeric_cols=['player_age', 'player_number'],
    categorical_cols=['player_position', 'player_name'],
    date_cols=None,
    drop_cols=['player_photo'],  # Non-critical
    drop_duplicates=False
)

# 5. df_standings_raw
df_standings_raw = clean_dataframe(
    df_standings_raw, "df_standings_raw",
    numeric_cols=['points', 'goal_diff', 'matches_played', 'wins', 'draws', 'losses', 
                  'goals_for', 'goals_against', 'points_per_game', 'win_percentage', 
                  'goals_per_game', 'goals_conceded_per_game'],
    categorical_cols=['group', 'form', 'status', 'description'],
    date_cols=['update_date'],
    drop_cols=None,
    drop_duplicates=False
)

# 6. df_league_teams_raw
df_league_teams_raw = clean_dataframe(
    df_league_teams_raw, "df_league_teams_raw",
    numeric_cols=['matches_played_home', 'matches_played_away', 'wins_home', 'wins_away', 
                  'draws_home', 'draws_away', 'losses_home', 'losses_away', 'goals_for_home', 
                  'goals_for_away', 'goals_against_home', 'goals_against_away', 
                  'clean_sheets_home', 'clean_sheets_away', 'failed_to_score_home', 
                  'failed_to_score_away', 'biggest_streak_wins', 'biggest_streak_draws', 
                  'biggest_streak_loses'],
    categorical_cols=['form', 'biggest_wins_home', 'biggest_wins_away', 'biggest_loses_home', 
                     'biggest_loses_away'],
    date_cols=None,
    drop_cols=None,
    drop_duplicates=False
)

# 7. df_h2h_raw
df_h2h_raw = clean_dataframe(
    df_h2h_raw, "df_h2h_raw",
    numeric_cols=['home_goals', 'away_goals', 'home_score_halftime', 'away_score_halftime', 
                  'home_score_fulltime', 'away_score_fulltime', 'h2h_total_matches', 
                  'h2h_home_wins', 'h2h_draws', 'h2h_away_wins', 'h2h_home_win_pct', 
                  'h2h_draw_pct', 'h2h_away_win_pct', 'h2h_avg_home_goals', 
                  'h2h_avg_away_goals', 'h2h_avg_total_goals', 'h2h_btts_pct', 
                  'h2h_over_2_5_pct', 'h2h_last_5_home_wins', 'h2h_last_5_draws', 
                  'h2h_last_5_away_wins', 'h2h_days_since_last_meeting'],
    categorical_cols=['venue_name', 'venue_city', 'referee', 'status', 'h2h_last_result'],
    date_cols=['date'],
    drop_cols=None,
    drop_duplicates=False
)

# 8. df_fixture_stats_raw
df_fixture_stats_raw = clean_dataframe(
    df_fixture_stats_raw, "df_fixture_stats_raw",
    numeric_cols=['home_shots_on_goal', 'away_shots_on_goal', 'home_total_shots', 
                  'away_total_shots', 'home_fouls', 'away_fouls', 'home_corners', 
                  'away_corners', 'home_yellow_cards', 'away_yellow_cards', 
                  'home_passes_accurate', 'away_passes_accurate'],
    categorical_cols=['home_team_name', 'away_team_name'],
    date_cols=['date'],
    drop_cols=['home_possession', 'away_possession', 'home_passes_percent', 'away_passes_percent'],  # Object type, needs parsing
    drop_duplicates=False
)

# 9. df_lineups_raw
df_lineups_raw = clean_dataframe(
    df_lineups_raw, "df_lineups_raw",
    numeric_cols=['home_startXI_count', 'home_subs_count', 'away_startXI_count', 'away_subs_count'],
    categorical_cols=['home_formation', 'home_coach_name', 'away_formation', 'away_coach_name'],
    date_cols=['date'],
    drop_cols=['home_coach_photo', 'home_startXI_players', 'away_coach_photo', 'away_startXI_players'],  # High missingness
    drop_duplicates=False
)

# Function to re-run quality check (same as before)
def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = df.columns[:5]
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    


Cleaning df_top_players_raw...
\n Quality Check for df_top_players_raw 

Shape: (21000, 22)
\nData Types:
season                int64
league               object
player_id             int64
player_name          object
age                 float64
nationality          object
team                 object
position             object
minutes             float64
rating              float64
goals_total         float64
assists             float64
shots_total         float64
shots_on            float64
passes_total        float64
key_passes          float64
tackles_total       float64
interceptions       float64
duels_total         float64
duels_won           float64
dribbles_success    float64
fouls_committed     float64
dtype: object

\nMissing Values per Column:
season              0
league              0
player_id           0
player_name         0
age                 0
nationality         0
team                0
position            0
minutes             0
rating              0
goals_total  

### Re-run quality check for df_top_players_raw

In [49]:
# Re-run quality check for df_top_players_raw
def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['season', 'league', 'player_id', 'player_name', 'team']
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

data_quality_check(df_top_players_raw, "df_top_players_raw")


 Quality Check for df_top_players_raw 

Shape: (21000, 22)

Data Types:
season                int64
league               object
player_id             int64
player_name          object
age                 float64
nationality          object
team                 object
position             object
minutes             float64
rating              float64
goals_total         float64
assists             float64
shots_total         float64
shots_on            float64
passes_total        float64
key_passes          float64
tackles_total       float64
interceptions       float64
duels_total         float64
duels_won           float64
dribbles_success    float64
fouls_committed     float64
dtype: object


Missing Values per Column:
season              0
league              0
player_id           0
player_name         0
age                 0
nationality         0
team                0
position            0
minutes             0
rating              0
goals_total         0
assists             0
shot

In [50]:
#  Refine Cleaning for df_top_players_raw 

# 1. Fix invalid age (replace 0 with median)
median_age = df_top_players_raw[df_top_players_raw['age'] > 0]['age'].median()
df_top_players_raw.loc[df_top_players_raw['age'] == 0, 'age'] = median_age

# 2. Impute 0 for stats of players with low minutes (<90)
low_minute_threshold = 90
numeric_stats = ['goals_total', 'assists', 'shots_total', 'shots_on', 'passes_total', 
                 'key_passes', 'tackles_total', 'interceptions', 'duels_total', 
                 'duels_won', 'dribbles_success', 'fouls_committed']
for col in numeric_stats:
    df_top_players_raw.loc[df_top_players_raw['minutes'] < low_minute_threshold, col] = 0

# 3. Convert count columns to int32
count_cols = ['goals_total', 'assists', 'shots_total', 'shots_on', 'passes_total', 
              'key_passes', 'tackles_total', 'interceptions', 'duels_total', 
              'duels_won', 'dribbles_success', 'fouls_committed']
for col in count_cols:
    df_top_players_raw[col] = df_top_players_raw[col].astype('int32')

# Re-run quality check
def data_quality_check(df, name):
    print(f"\n Quality Check for {name} (Refined) \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['season', 'league', 'player_id', 'player_name', 'team']
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

data_quality_check(df_top_players_raw, "df_top_players_raw")


 Quality Check for df_top_players_raw (Refined) 

Shape: (21000, 22)

Data Types:
season                int64
league               object
player_id             int64
player_name          object
age                 float64
nationality          object
team                 object
position             object
minutes             float64
rating              float64
goals_total           int32
assists               int32
shots_total           int32
shots_on              int32
passes_total          int32
key_passes            int32
tackles_total         int32
interceptions         int32
duels_total           int32
duels_won             int32
dribbles_success      int32
fouls_committed       int32
dtype: object


Missing Values per Column:
season              0
league              0
player_id           0
player_name         0
age                 0
nationality         0
team                0
position            0
minutes             0
rating              0
goals_total         0
assists         

### Clean df_injuries_raw 

In [51]:
#  Clean df_injuries_raw 

def clean_dataframe(df, name, numeric_cols=None, categorical_cols=None, date_cols=None, drop_cols=None, drop_duplicates=False):
    print(f"\nCleaning {name}...")
    df = df.copy()
    
    # Drop specified columns
    if drop_cols:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
    
    # Convert dates
    if date_cols:
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Impute numerics with median
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
    
    # Impute categoricals with mode or 'Unknown'
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    # Drop duplicates if specified
    if drop_duplicates:
        df.drop_duplicates(inplace=True)
    
    # Re-run quality check
    data_quality_check(df, name)
    return df

def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = df.columns[:5]
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

# Clean df_injuries_raw
df_injuries_raw = clean_dataframe(
    df_injuries_raw, "df_injuries_raw",
    numeric_cols=['player_id', 'fixture_id'],  
    categorical_cols=['player_name', 'injury_reason', 'team', 'league'],
    date_cols=['fixture_date'],
    drop_cols=['age'],  
    drop_duplicates=True
)


Cleaning df_injuries_raw...

 Quality Check for df_injuries_raw 

Shape: (37196, 8)

Data Types:
season                         int64
league                        object
player_id                      int64
player_name                   object
team                          object
fixture_id                     int64
fixture_date     datetime64[ns, UTC]
injury_reason                 object
dtype: object


Missing Values per Column:
season           0
league           0
player_id        0
player_name      0
team             0
fixture_id       0
fixture_date     0
injury_reason    0
dtype: int64

Total Missing Values: 0 (0.00% of data)


Number of Duplicates: 0


Unique Values in Key Columns:
season: 3 unique values
league: 9 unique values
player_id: 4300 unique values
player_name: 4270 unique values
team: 210 unique values

Descriptive Statistics for Numeric Columns:
         season  player_id  fixture_id
count 37196.000  37196.000   37196.000
mean   2021.353  45259.078  786254.173
std

### Clean df_squad_basic_raw 

In [52]:
#  Clean df_squad_basic_raw 

def clean_dataframe(df, name, numeric_cols=None, categorical_cols=None, date_cols=None, drop_cols=None, drop_duplicates=False):
    print(f"\nCleaning {name}...")
    df = df.copy()
    
    # Drop specified columns
    if drop_cols:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
    
    # Convert dates
    if date_cols:
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Impute numerics with median
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
    
    # Impute categoricals with mode or 'Unknown'
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    # Drop duplicates if specified
    if drop_duplicates:
        df.drop_duplicates(inplace=True)
    
    # Re-run quality check
    data_quality_check(df, name)
    return df

def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = df.columns[:5]
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

# Clean df_squad_basic_raw
df_squad_basic_raw = clean_dataframe(
    df_squad_basic_raw, "df_squad_basic_raw",
    numeric_cols=['player_age', 'player_number'],
    categorical_cols=['player_position', 'player_name', 'team_name', 'league_name'],
    date_cols=None,
    drop_cols=['player_photo'],  # Non-critical
    drop_duplicates=False
)


Cleaning df_squad_basic_raw...

 Quality Check for df_squad_basic_raw 

Shape: (78547, 10)

Data Types:
season               int64
league_id            int64
league_name         object
team_id              int64
team_name           object
player_id            int64
player_name         object
player_age         float64
player_number      float64
player_position     object
dtype: object


Missing Values per Column:
season             0
league_id          0
league_name        0
team_id            0
team_name          0
player_id          0
player_name        0
player_age         0
player_number      0
player_position    0
dtype: int64

Total Missing Values: 0 (0.00% of data)


Number of Duplicates: 0


Unique Values in Key Columns:
season: 6 unique values
league_id: 10 unique values
league_name: 10 unique values
team_id: 569 unique values
team_name: 568 unique values

Descriptive Statistics for Numeric Columns:
         season  league_id   team_id  player_id  player_age  player_number
co

In [53]:
#  Refine Cleaning for df_squad_basic_raw 

# Cap player_age at 50
max_age = 50
median_age = df_squad_basic_raw[df_squad_basic_raw['player_age'] <= max_age]['player_age'].median()
df_squad_basic_raw.loc[df_squad_basic_raw['player_age'] > max_age, 'player_age'] = median_age

# Re-run quality check
def data_quality_check(df, name):
    print(f"\n Quality Check for {name} (Refined) \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['season', 'league_id', 'league_name', 'team_id', 'team_name']
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

data_quality_check(df_squad_basic_raw, "df_squad_basic_raw")


 Quality Check for df_squad_basic_raw (Refined) 

Shape: (78547, 10)

Data Types:
season               int64
league_id            int64
league_name         object
team_id              int64
team_name           object
player_id            int64
player_name         object
player_age         float64
player_number      float64
player_position     object
dtype: object


Missing Values per Column:
season             0
league_id          0
league_name        0
team_id            0
team_name          0
player_id          0
player_name        0
player_age         0
player_number      0
player_position    0
dtype: int64

Total Missing Values: 0 (0.00% of data)


Number of Duplicates: 0


Unique Values in Key Columns:
season: 6 unique values
league_id: 10 unique values
league_name: 10 unique values
team_id: 569 unique values
team_name: 568 unique values

Descriptive Statistics for Numeric Columns:
         season  league_id   team_id  player_id  player_age  player_number
count 78547.000  78547.0

### Clean df_standings_raw 

In [54]:
#  Clean df_standings_raw 

def clean_dataframe(df, name, numeric_cols=None, categorical_cols=None, date_cols=None, drop_cols=None, drop_duplicates=False):
    print(f"\nCleaning {name}...")
    df = df.copy()
    
    # Drop specified columns
    if drop_cols:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
    
    # Convert dates
    if date_cols:
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Impute numerics with median
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
    
    # Impute categoricals with mode or 'Unknown'
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    # Drop duplicates if specified
    if drop_duplicates:
        df.drop_duplicates(inplace=True)
    
    # Re-run quality check
    data_quality_check(df, name)
    return df

def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['season', 'league_id', 'league_name', 'team_id', 'team_name']
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

# Clean df_standings_raw
df_standings_raw = clean_dataframe(
    df_standings_raw, "df_standings_raw",
    numeric_cols=['rank', 'points', 'goal_diff', 'matches_played', 'wins', 'draws', 'losses', 
                  'goals_for', 'goals_against', 'matches_played_home', 'wins_home', 
                  'draws_home', 'losses_home', 'goals_for_home', 'goals_against_home', 
                  'matches_played_away', 'wins_away', 'draws_away', 'losses_away', 
                  'goals_for_away', 'goals_against_away', 'points_per_game', 
                  'win_percentage', 'goals_per_game', 'goals_conceded_per_game'],
    categorical_cols=['group', 'form', 'status', 'description'],
    date_cols=['update_date'],
    drop_cols=None,
    drop_duplicates=False
)


Cleaning df_standings_raw...

 Quality Check for df_standings_raw 

Shape: (862, 35)

Data Types:
season                                   int64
league_id                                int64
league_name                             object
team_id                                  int64
team_name                               object
rank                                     int64
points                                   int64
goal_diff                                int64
group                                   object
form                                    object
status                                  object
description                             object
matches_played                           int64
wins                                     int64
draws                                    int64
losses                                   int64
goals_for                                int64
goals_against                            int64
matches_played_home                      int64
wins_hom

### Clean df_league_teams_raw 

In [55]:
#  Clean df_league_teams_raw 

def clean_dataframe(df, name, numeric_cols=None, categorical_cols=None, date_cols=None, drop_cols=None, drop_duplicates=False):
    print(f"\nCleaning {name}...")
    df = df.copy()
    
    # Drop specified columns
    if drop_cols:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
    
    # Convert dates
    if date_cols:
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Impute numerics with median
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
    
    # Impute categoricals with mode or 'Unknown'
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    # Drop duplicates if specified
    if drop_duplicates:
        df.drop_duplicates(inplace=True)
    
    # Re-run quality check
    data_quality_check(df, name)
    return df

def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['team_id', 'team_name', 'league_id', 'league_name', 'season']
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

# Clean df_league_teams_raw
df_league_teams_raw = clean_dataframe(
    df_league_teams_raw, "df_league_teams_raw",
    numeric_cols=['matches_played_home', 'matches_played_away', 'matches_played_total', 
                  'wins_home', 'wins_away', 'wins_total', 'draws_home', 'draws_away', 
                  'draws_total', 'losses_home', 'losses_away', 'losses_total', 
                  'goals_for_home', 'goals_for_away', 'goals_for_total', 
                  'goals_for_avg_home', 'goals_for_avg_away', 'goals_for_avg_total', 
                  'goals_against_home', 'goals_against_against_total', 
                  'goals_against_avg_home', 'goals_against_avg_away', 
                  'goals_against_avg_total', 'clean_sheets_home', 'clean_sheets_away', 
                  'clean_sheets_total', 'failed_to_score_home', 'failed_to_score_away', 
                  'failed_to_score_total', 'biggest_streak_wins', 
                  'biggest_streak_draws', 'biggest_streak_loses'],
    categorical_cols=['team_name', 'league_name', 'form', 'biggest_wins_home', 
                     'biggest_wins_away', 'biggest_loses_home', 'biggest_loses_away'],
    date_cols=None,
    drop_cols=None,
    drop_duplicates=False
)


Cleaning df_league_teams_raw...

 Quality Check for df_league_teams_raw 

Shape: (2659, 43)

Data Types:
team_id                      int64
team_name                   object
league_id                    int64
league_name                 object
season                       int64
matches_played_home          int64
matches_played_away          int64
matches_played_total         int64
wins_home                    int64
wins_away                    int64
wins_total                   int64
draws_home                   int64
draws_away                   int64
draws_total                  int64
losses_home                  int64
losses_away                  int64
losses_total                 int64
goals_for_home               int64
goals_for_away               int64
goals_for_total              int64
goals_for_avg_home         float64
goals_for_avg_away         float64
goals_for_avg_total        float64
goals_against_home           int64
goals_against_away           int64
goals_against_total

### Clean df_h2h_raw 

In [56]:
#  Clean df_h2h_raw 

def clean_dataframe(df, name, numeric_cols=None, categorical_cols=None, date_cols=None, drop_cols=None, drop_duplicates=False):
    print(f"\nCleaning {name}...")
    df = df.copy()
    
    # Drop specified columns
    if drop_cols:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
    
    # Convert dates
    if date_cols:
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Impute numerics with median
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
    
    # Impute categoricals with mode or 'Unknown'
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    # Drop duplicates if specified
    if drop_duplicates:
        df.drop_duplicates(inplace=True)
    
    # Re-run quality check
    data_quality_check(df, name)
    return df

def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['fixture_id', 'date', 'league_id', 'home_team_id', 'away_team_id']
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

# Clean df_h2h_raw
df_h2h_raw = clean_dataframe(
    df_h2h_raw, "df_h2h_raw",
    numeric_cols=['home_goals', 'away_goals', 'home_score_halftime', 'away_score_halftime', 
                  'home_score_fulltime', 'away_score_fulltime', 'h2h_total_matches', 
                  'h2h_home_wins', 'h2h_draws', 'h2h_away_wins', 'h2h_home_win_pct', 
                  'h2h_draw_pct', 'h2h_away_win_pct', 'h2h_avg_home_goals', 
                  'h2h_avg_away_goals', 'h2h_avg_total_goals', 'h2h_btts_pct', 
                  'h2h_over_2_5_pct', 'h2h_last_5_home_wins', 'h2h_last_5_draws', 
                  'h2h_last_5_away_wins', 'h2h_days_since_last_meeting'],
    categorical_cols=['league_name', 'round', 'home_team_name', 'away_team_name', 
                     'venue_name', 'venue_city', 'referee', 'status', 'h2h_last_result'],
    date_cols=['date'],
    drop_cols=None,
    drop_duplicates=False
)


Cleaning df_h2h_raw...

 Quality Check for df_h2h_raw 

Shape: (18894, 38)

Data Types:
fixture_id                                   int64
date                           datetime64[ns, UTC]
timestamp                                    int64
league_id                                    int64
league_name                                 object
season                                       int64
round                                       object
home_team_id                                 int64
home_team_name                              object
away_team_id                                 int64
away_team_name                              object
home_goals                                   int64
away_goals                                   int64
home_score_halftime                        float64
away_score_halftime                        float64
home_score_fulltime                          int64
away_score_fulltime                          int64
venue_name                                  

### Clean df_fixture_stats_raw 

In [57]:
#  Clean df_fixture_stats_raw 

def clean_dataframe(df, name, numeric_cols=None, categorical_cols=None, date_cols=None, drop_cols=None, drop_duplicates=False):
    print(f"\nCleaning {name}...")
    df = df.copy()
    
    # Drop specified columns
    if drop_cols:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
    
    # Convert dates
    if date_cols:
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Impute numerics with median
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
    
    # Impute categoricals with mode or 'Unknown'
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    # Drop duplicates if specified
    if drop_duplicates:
        df.drop_duplicates(inplace=True)
    
    # Re-run quality check
    data_quality_check(df, name)
    return df

def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['fixture_id', 'date', 'league_id', 'home_team_name', 'away_team_name']
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

# Clean df_fixture_stats_raw
df_fixture_stats_raw = clean_dataframe(
    df_fixture_stats_raw, "df_fixture_stats_raw",
    numeric_cols=['home_shots_on_goal', 'away_shots_on_goal', 'home_total_shots', 
                  'away_total_shots', 'home_fouls', 'away_fouls', 'home_corners', 
                  'away_corners', 'home_yellow_cards', 'away_yellow_cards', 
                  'home_passes_accurate', 'away_passes_accurate'],
    categorical_cols=['home_team_name', 'away_team_name'],
    date_cols=['date'],
    drop_cols=['home_possession', 'away_possession', 'home_passes_percent', 'away_passes_percent'],
    drop_duplicates=False
)


Cleaning df_fixture_stats_raw...

 Quality Check for df_fixture_stats_raw 

Shape: (17628, 21)

Data Types:
fixture_id                            int64
league_id                             int64
league_name                          object
season                                int64
date                    datetime64[ns, UTC]
home_team_id                          int64
home_team_name                       object
away_team_id                          int64
away_team_name                       object
home_shots_on_goal                  float64
away_shots_on_goal                  float64
home_total_shots                    float64
away_total_shots                    float64
home_fouls                          float64
away_fouls                          float64
home_corners                        float64
away_corners                        float64
home_yellow_cards                   float64
away_yellow_cards                   float64
home_passes_accurate                float64
away_passes

### Clean df_lineups_raw 

In [58]:
#  Clean df_lineups_raw 

def clean_dataframe(df, name, numeric_cols=None, categorical_cols=None, date_cols=None, drop_cols=None, drop_duplicates=False):
    print(f"\nCleaning {name}...")
    df = df.copy()
    
    # Drop specified columns
    if drop_cols:
        df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
    
    # Convert dates
    if date_cols:
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Impute numerics with median
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                df[col].fillna(df[col].median(), inplace=True)
    
    # Impute categoricals with mode or 'Unknown'
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown', inplace=True)
    
    # Drop duplicates if specified
    if drop_duplicates:
        df.drop_duplicates(inplace=True)
    
    # Re-run quality check
    data_quality_check(df, name)
    return df

def data_quality_check(df, name):
    print(f"\n Quality Check for {name} \n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['fixture_id', 'date', 'home_team_id', 'home_team_name', 'away_team_id']
    for col in key_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

# Clean df_lineups_raw
df_lineups_raw = clean_dataframe(
    df_lineups_raw, "df_lineups_raw",
    numeric_cols=['fixture_id', 'league_id', 'season', 'home_team_id', 'away_team_id', 
                  'home_coach_id', 'away_coach_id', 'home_startXI_count', 
                  'home_subs_count', 'away_startXI_count', 'away_subs_count'],
    categorical_cols=['league_name', 'home_team_name', 'away_team_name', 
                     'home_formation', 'home_coach_name', 'away_formation', 
                     'away_coach_name'],
    date_cols=['date'],
    drop_cols=None,
    drop_duplicates=False
)


Cleaning df_lineups_raw...

 Quality Check for df_lineups_raw 

Shape: (18011, 19)

Data Types:
fixture_id                          int64
league_id                           int64
league_name                        object
season                              int64
date                  datetime64[ns, UTC]
home_team_id                        int64
home_team_name                     object
away_team_id                        int64
away_team_name                     object
home_formation                     object
home_coach_id                     float64
home_coach_name                    object
home_startXI_count                  int64
home_subs_count                     int64
away_formation                     object
away_coach_id                     float64
away_coach_name                    object
away_startXI_count                  int64
away_subs_count                     int64
dtype: object


Missing Values per Column:
fixture_id            0
league_id             0
league_name   

In [59]:
# Aggregate, Merge, Impute, and Feature Engineering with Full Mapping

# Step 1: Create Clean team_mapping from df_fixtures
home_teams = df_fixtures[['home_team_id', 'home_team_name']].rename(columns={
    'home_team_id': 'team_id',
    'home_team_name': 'team_name'
})
away_teams = df_fixtures[['away_team_id', 'away_team_name']].rename(columns={
    'away_team_id': 'team_id',
    'away_team_name': 'team_name'
})
team_mapping_df = pd.concat([home_teams, away_teams]).drop_duplicates()

duplicate_team_names = team_mapping_df[team_mapping_df['team_name'].duplicated(keep=False)]
if not duplicate_team_names.empty:
    print("Duplicate team_name entries found in df_fixtures:")
    print(duplicate_team_names)
    team_mapping_df = team_mapping_df.drop_duplicates(subset='team_name', keep='first')
else:
    print("No duplicate team_name entries in df_fixtures.")

team_mapping = team_mapping_df.set_index('team_name')['team_id']


Duplicate team_name entries found in df_fixtures:
       team_id team_name
10810     2248     Drita
11227    14281     Drita


In [60]:

# Step 2: Fix team column types for df_top_players_raw
print("\nSample of df_top_players_raw['team']:")
print(df_top_players_raw['team'].head(10))
print("dtype:", df_top_players_raw['team'].dtype)
print("Unique values sample:", df_top_players_raw['team'].unique()[:10])

df_top_players_raw['team_id'] = pd.to_numeric(df_top_players_raw['team'], errors='coerce')
unmapped_numeric = df_top_players_raw[df_top_players_raw['team_id'].isna()]['team'].unique()
if len(unmapped_numeric) > 0:
    print(f"Warning: {len(unmapped_numeric)} non-numeric teams in df_top_players_raw, trying name mapping:")
    print(unmapped_numeric)
    df_top_players_raw.loc[df_top_players_raw['team_id'].isna(), 'team_id'] = (
        df_top_players_raw.loc[df_top_players_raw['team_id'].isna(), 'team'].map(team_mapping)
    )

unmapped_top_players = df_top_players_raw[df_top_players_raw['team_id'].isna()]['team'].unique()
if len(unmapped_top_players) > 0:
    print(f"Warning: {len(unmapped_top_players)} unmapped teams in df_top_players_raw:")
    print(unmapped_top_players)
    df_top_players_raw['team_id'] = df_top_players_raw['team_id'].fillna(-1).astype('int64')
else:
    df_top_players_raw['team_id'] = df_top_players_raw['team_id'].astype('int64')

# Step 2: Fix team column types for df_injuries_raw
print("\nSample of df_injuries_raw['team']:")
print(df_injuries_raw['team'].head(10))
print("dtype:", df_injuries_raw['team'].dtype)
print("Unique values sample:", df_injuries_raw['team'].unique()[:10])

df_injuries_raw['team_id'] = pd.to_numeric(df_injuries_raw['team'], errors='coerce')
unmapped_numeric_injuries = df_injuries_raw[df_injuries_raw['team_id'].isna()]['team'].unique()
if len(unmapped_numeric_injuries) > 0:
    print(f"Warning: {len(unmapped_numeric_injuries)} non-numeric teams in df_injuries_raw, trying name mapping:")
    print(unmapped_numeric_injuries)
    df_injuries_raw.loc[df_injuries_raw['team_id'].isna(), 'team_id'] = (
        df_injuries_raw.loc[df_injuries_raw['team_id'].isna(), 'team'].map(team_mapping)
    )

unmapped_injuries = df_injuries_raw[df_injuries_raw['team_id'].isna()]['team'].unique()
if len(unmapped_injuries) > 0:
    print(f"Warning: {len(unmapped_injuries)} unmapped teams in df_injuries_raw:")
    print(unmapped_injuries)
    df_injuries_raw['team_id'] = df_injuries_raw['team_id'].fillna(-1).astype('int64')
else:
    df_injuries_raw['team_id'] = df_injuries_raw['team_id'].astype('int64')



Sample of df_top_players_raw['team']:
0    Wolfsberger AC
1         Feyenoord
2            Getafe
3     Saint Etienne
4            Getafe
5         FC Astana
6          Espanyol
7         FC Lugano
8     CFR 1907 Cluj
9         FC Lugano
Name: team, dtype: object
dtype: object
Unique values sample: ['Wolfsberger AC' 'Feyenoord' 'Getafe' 'Saint Etienne' 'FC Astana'
 'Espanyol' 'FC Lugano' 'CFR 1907 Cluj' 'FC Basel 1893' 'Gent']
['Wolfsberger AC' 'Feyenoord' 'Getafe' 'Saint Etienne' 'FC Astana'
 'Espanyol' 'FC Lugano' 'CFR 1907 Cluj' 'FC Basel 1893' 'Gent'
 'FK Partizan' 'Ferencvarosi TC' 'UE Engordany' 'La Fiorita' 'Tre Penne'
 'Tre Fiori' 'Feronikeli' 'UE Sant Julia' 'FC Santa Coloma' 'SC Braga'
 'KI Klaksvik' 'NSI Runavik' 'HB' 'Speranţa Nisporeni' 'Zeta' 'FK Kukesi'
 'Chikhura Sachkhere' 'B36 Torshavn' 'Partizani' 'Laci' 'Debreceni VSC'
 'KR Reykjavik' 'Piast Gliwice' 'FC Thun' 'KuPS' 'Inter Turku' 'Rops'
 'Sutjeska' 'Shkupi 1927' 'FC Luzern' 'Sturm Graz' 'Budapest Honved'
 'Utrecht

In [61]:

# Step 3: Filter out unmapped teams
df_top_players_raw = df_top_players_raw[df_top_players_raw['team_id'] != -1]
df_injuries_raw = df_injuries_raw[df_injuries_raw['team_id'] != -1]

In [62]:

# Step 4: Aggregate Team-Level Datasets
df_top_players_agg = df_top_players_raw.groupby(['team_id', 'season']).agg({
    'rating': 'mean',
    'goals_total': 'sum',
    'assists': 'sum',
    'minutes': 'sum'
}).reset_index().rename(columns={
    'rating': 'avg_team_rating',
    'goals_total': 'total_team_goals',
    'assists': 'total_team_assists',
    'minutes': 'total_team_minutes'
})

df_injuries_agg = df_injuries_raw.groupby(['team_id', 'season']).agg({
    'player_id': 'count'
}).reset_index().rename(columns={'player_id': 'injury_count'})

df_squad_basic_agg = df_squad_basic_raw.groupby(['team_id', 'season']).agg({
    'player_id': 'count',
    'player_age': 'mean'
}).reset_index().rename(columns={'player_id': 'squad_size', 'player_age': 'avg_squad_age'})

df_standings_agg = df_standings_raw.groupby(['team_id', 'season']).agg({
    'rank': 'mean',
    'points': 'sum',
    'goal_diff': 'sum',
    'win_percentage': 'mean'
}).reset_index().rename(columns={
    'rank': 'avg_rank',
    'points': 'total_points',
    'goal_diff': 'total_goal_diff',
    'win_percentage': 'avg_win_percentage'
})

df_league_teams_agg = df_league_teams_raw.groupby(['team_id', 'season']).agg({
    'wins_total': 'sum',
    'goals_for_total': 'sum',
    'goals_against_total': 'sum',
    'clean_sheets_total': 'sum'
}).reset_index().rename(columns={
    'wins_total': 'total_wins',
    'goals_for_total': 'total_goals_for',
    'goals_against_total': 'total_goals_against',
    'clean_sheets_total': 'total_clean_sheets'
})


In [63]:

# Step 5: Merge All Datasets
df_merged = df_fixtures.copy()

df_merged = df_merged.merge(df_h2h_raw, on=['fixture_id'], how='left', suffixes=('', '_h2h'))
df_merged = df_merged.merge(df_fixture_stats_raw, on=['fixture_id'], how='left', suffixes=('', '_stats'))
df_merged = df_merged.merge(df_lineups_raw, on=['fixture_id'], how='left', suffixes=('', '_lineups'))

# Merge home team aggregates
df_merged = df_merged.merge(df_top_players_agg, left_on=['home_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])
df_merged = df_merged.merge(df_injuries_agg, left_on=['home_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])
df_merged = df_merged.merge(df_squad_basic_agg, left_on=['home_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])
df_merged = df_merged.merge(df_standings_agg, left_on=['home_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])
df_merged = df_merged.merge(df_league_teams_agg, left_on=['home_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])

# Merge away team aggregates
df_merged = df_merged.merge(df_top_players_agg.rename(columns={
    'avg_team_rating': 'away_avg_team_rating',
    'total_team_goals': 'away_total_team_goals',
    'total_team_assists': 'away_total_team_assists',
    'total_team_minutes': 'away_total_team_minutes'
}), left_on=['away_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])

df_merged = df_merged.merge(df_injuries_agg.rename(columns={'injury_count': 'away_injury_count'}), left_on=['away_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])

df_merged = df_merged.merge(df_squad_basic_agg.rename(columns={
    'squad_size': 'away_squad_size',
    'avg_squad_age': 'away_avg_squad_age'
}), left_on=['away_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])

df_merged = df_merged.merge(df_standings_agg.rename(columns={
    'avg_rank': 'away_avg_rank',
    'total_points': 'away_total_points',
    'total_goal_diff': 'away_total_goal_diff',
    'avg_win_percentage': 'away_avg_win_percentage'
}), left_on=['away_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])

df_merged = df_merged.merge(df_league_teams_agg.rename(columns={
    'total_wins': 'away_total_wins',
    'total_goals_for': 'away_total_goals_for',
    'total_goals_against': 'away_total_goals_against',
    'total_clean_sheets': 'away_total_clean_sheets'
}), left_on=['away_team_id', 'season'], right_on=['team_id', 'season'], how='left').drop(columns=['team_id'])

In [64]:

# Step 6: Impute Missing Values
numeric_cols = [
    'home_shots_on_goal', 'away_shots_on_goal', 'home_total_shots', 'away_total_shots',
    'home_fouls', 'away_fouls', 'home_corners', 'away_corners',
    'home_yellow_cards', 'away_yellow_cards', 'home_passes_accurate', 'away_passes_accurate',
    'avg_team_rating', 'total_team_goals', 'total_team_assists', 'total_team_minutes',
    'away_avg_team_rating', 'away_total_team_goals', 'away_total_team_assists', 'away_total_team_minutes',
    'injury_count', 'away_injury_count', 'squad_size', 'avg_squad_age',
    'away_squad_size', 'away_avg_squad_age', 'avg_rank', 'total_points', 'total_goal_diff',
    'avg_win_percentage', 'away_avg_rank', 'away_total_points', 'away_total_goal_diff',
    'away_avg_win_percentage', 'h2h_home_wins', 'h2h_draws', 'h2h_away_wins',
    'h2h_home_win_pct', 'h2h_draw_pct', 'h2h_away_win_pct', 'h2h_avg_home_goals',
    'h2h_avg_away_goals', 'h2h_avg_total_goals', 'h2h_btts_pct', 'h2h_over_2_5_pct',
    'h2h_last_5_home_wins', 'h2h_last_5_draws', 'h2h_last_5_away_wins', 'h2h_days_since_last_meeting'
]
for col in numeric_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged.groupby('league_id')[col].transform(lambda x: x.fillna(x.median()))

df_merged['injury_count'] = df_merged['injury_count'].fillna(0)
df_merged['away_injury_count'] = df_merged['away_injury_count'].fillna(0)

categorical_cols = ['home_coach_id', 'away_coach_id', 'home_formation', 'away_formation']
for col in categorical_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged.groupby('home_team_id')[col].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else x))

missing_pct = df_merged.isna().mean()
cols_to_drop = missing_pct[missing_pct > 0.8].index
if len(cols_to_drop) > 0:
    print(f"Dropping columns with >80% missing values: {list(cols_to_drop)}")
    df_merged = df_merged.drop(columns=cols_to_drop)

In [65]:
# Step 7: Feature Engineering
df_merged['h2h_rating_diff'] = df_merged['avg_team_rating'] - df_merged['away_avg_team_rating']
df_merged['h2h_points_diff'] = df_merged['total_points'] - df_merged['away_total_points']
df_merged['h2h_injury_diff'] = df_merged['injury_count'] - df_merged['away_injury_count']

df_merged['home_rating_rank_ratio'] = df_merged['avg_team_rating'] / (df_merged['avg_rank'] + 1)
df_merged['away_rating_rank_ratio'] = df_merged['away_avg_team_rating'] / (df_merged['away_avg_rank'] + 1)
df_merged['home_points_per_game'] = df_merged['total_points'] / (df_merged['total_wins'] + df_merged['h2h_draws'] + df_merged['h2h_away_wins'] + 1)
df_merged['away_points_per_game'] = df_merged['away_total_points'] / (df_merged['total_wins'] + df_merged['h2h_draws'] + df_merged['h2h_away_wins'] + 1)


In [67]:

# Quality Check
def data_quality_check(df, name):
    print(f"\nQuality Check for {name}\n")
    print(f"Shape: {df.shape}")
    print(f"\nData Types:\n{df.dtypes}\n")
    print(f"\nMissing Values per Column:\n{df.isna().sum()}\n")
    total_missing = df.isna().sum().sum()
    print(f"Total Missing Values: {total_missing} ({total_missing / (df.shape[0] * df.shape[1]) * 100:.2f}% of data)\n")
    print(f"\nNumber of Duplicates: {df.duplicated().sum()}\n")
    print(f"\nUnique Values in Key Columns:")
    key_cols = ['fixture_id', 'date', 'league_id', 'home_team_id', 'away_team_id']
    for col in key_cols:
        if col in df.columns:
            print(f"{col}: {df[col].nunique()} unique values")
    print(f"\nDescriptive Statistics for Numeric Columns:\n{df.describe()}\n")
    print("\n")

data_quality_check(df_merged, "df_merged")

# Save Cleaned Data
df_merged.to_csv(FEATURES_PATH/'df_merged_cleaned.csv', index=False)
print(f"Saved cleaned dataset to '{FEATURES_PATH}/df_merged_cleaned.csv'")


Quality Check for df_merged

Shape: (18894, 144)

Data Types:
fixture_id                              int64
date                      datetime64[ns, UTC]
timestamp                               int64
league_id                               int64
league_name                            object
                                 ...         
h2h_injury_diff                       float64
home_rating_rank_ratio                float64
away_rating_rank_ratio                float64
home_points_per_game                  float64
away_points_per_game                  float64
Length: 144, dtype: object


Missing Values per Column:
fixture_id                0
date                      0
timestamp                 0
league_id                 0
league_name               0
                         ..
h2h_injury_diff           0
home_rating_rank_ratio    0
away_rating_rank_ratio    0
home_points_per_game      0
away_points_per_game      0
Length: 144, dtype: int64

Total Missing Values: 22522 (0.83% of da